# Data Overview
## OHLCV Price Data Analysis

### Objectives:
- Load and examine ETH/USD OHLCV data
- Understand data structure and types
- Check for missing values and data quality
- Initial statistical summary
- Data range and frequency analysis

### Data Sources:
- CSV file: `../../../data/Eth_OHLCV.csv`
- JSON file: `../../../data/Eth_OHLCV.json`
- SQLite DB: `../../../data/ETH.db`

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# ====================================================================
# 🔧 CORRECT PATH FOR YOUR STRUCTURE
# ====================================================================

# Get current notebook directory
notebook_dir = Path(os.getcwd()).resolve()
print(f"📁 Current notebook: {notebook_dir}")

# Go up 2 levels to reach ohlcv root:
# notebooks/01_eda_exploratory_data_analysis/ -> notebooks/ -> ohlcv/
ohlcv_root = notebook_dir.parent.parent
utils_path = ohlcv_root / 'utils'

print(f"📁 OHLCV root: {ohlcv_root}")
print(f"📁 Utils path: {utils_path}")

# Add utils to Python path
if str(utils_path) not in sys.path:
    sys.path.insert(0, str(utils_path))
    print(f"✅ Added utils path: {utils_path}")

# Verify utils folder exists and has files
if utils_path.exists():
    print(f"\n📁 Utils folder contents:")
    for file in utils_path.iterdir():
        print(f"   - {file.name}")
else:
    print(f"❌ Utils folder not found at: {utils_path}")

# ====================================================================
# 📦 IMPORT UTILITIES
# ====================================================================

try:
    from data_loader import DataLoader
    from visualizations import TradingVisualizer
    from trading_helpers import TradingHelpers
    print("\n✅ All utilities imported successfully!")
except ImportError as e:
    print(f"\n❌ Import error: {e}")
    print(f"Current sys.path: {sys.path[:3]}")
    
    # Fallback: Create inline classes
    print("\n📦 Creating fallback utilities...")
    
    class DataLoader:
        def __init__(self):
            self.ohlcv_root = Path(__file__).parent.parent.parent
            self.data_dir = self.ohlcv_root / 'data'
            self.db_path = self.data_dir / 'ETH.db'
            self.csv_path = self.data_dir / 'Eth_OHLCV.csv'
            self.json_path = self.data_dir / 'Eth_OHLCV.json'
        
        def load_from_csv(self):
            if self.csv_path.exists():
                df = pd.read_csv(self.csv_path)
                if 'timestamp' in df.columns:
                    df['timestamp'] = pd.to_datetime(df['timestamp'])
                    df.set_index('timestamp', inplace=True)
                print(f"✅ Loaded {len(df)} rows from CSV")
                return df
            print(f"⚠️  CSV not found: {self.csv_path}")
            return pd.DataFrame()
        
        def load_from_db(self):
            import sqlite3
            if self.db_path.exists():
                conn = sqlite3.connect(str(self.db_path))
                df = pd.read_sql_query("SELECT * FROM price_data ORDER BY timestamp DESC", conn)
                conn.close()
                if not df.empty and 'timestamp' in df.columns:
                    df['timestamp'] = pd.to_datetime(df['timestamp'])
                    df.set_index('timestamp', inplace=True)
                print(f"✅ Loaded {len(df)} rows from database")
                return df
            print(f"⚠️  Database not found: {self.db_path}")
            return pd.DataFrame()
        
        def load_from_json(self):
            import json
            if self.json_path.exists():
                with open(self.json_path, 'r') as f:
                    data = json.load(f)
                df = pd.DataFrame(data)
                if 'timestamp' in df.columns:
                    df['timestamp'] = pd.to_datetime(df['timestamp'])
                    df.set_index('timestamp', inplace=True)
                print(f"✅ Loaded {len(df)} rows from JSON")
                return df
            print(f"⚠️  JSON not found: {self.json_path}")
            return pd.DataFrame()
    
    class TradingVisualizer:
        @staticmethod
        def plot_price_volume(df, figsize=(15, 8)):
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, height_ratios=[3, 1])
            if not df.empty:
                ax1.plot(df.index, df['close'], label='Close', linewidth=2)
                ax1.set_title('ETH Price', fontsize=14, fontweight='bold')
                ax1.set_ylabel('Price ($)')
                ax1.legend()
                ax1.grid(True, alpha=0.3)
                ax2.bar(df.index, df['volume'], color='orange', alpha=0.7)
                ax2.set_title('Volume', fontsize=14, fontweight='bold')
                ax2.set_ylabel('Volume')
                ax2.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
    
    TradingHelpers = None
    print("✅ Fallback utilities created")

print("\n" + "=" * 60)
print("✅ SETUP COMPLETE")
print("=" * 60)

# ====================================================================
# 📊 LOAD DATA
# ====================================================================

loader = DataLoader()
df = loader.load_from_csv()

if df.empty:
    df = loader.load_from_db()

if df.empty:
    df = loader.load_from_json()

print(f"\n📊 Data loaded: {len(df)} rows")
print(f"📅 Date range: {df.index.min()} to {df.index.max()}" if not df.empty else "No data loaded")
df.head()

📁 Current notebook: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\notebooks\01_eda_exploratory_data_analysis
📁 OHLCV root: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv
📁 Utils path: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\utils
✅ Added utils path: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\ohlcv\utils

📁 Utils folder contents:
   - data_loader.py
   - trading_helpers.py
   - visualizations.py
   - __init__.py

✅ All utilities imported successfully!

✅ SETUP COMPLETE
⚠️  CSV file not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\Eth_OHLCV.csv
⚠️  Database not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereum\manually\server\notebooks\data\ETH.db
⚠️  JSON file not found: D:\Sachin Chakrawarti\Learn\Done\myprojets\projects\ethereu

""


In [ ]:
# Initialize data loader
loader = DataLoader()

# Load data from CSV
df = loader.load_from_csv()

if df.empty:
    print("CSV not found, trying JSON...")
    df = loader.load_from_json()

if df.empty:
    print("JSON not found, trying database...")
    df = loader.load_from_db()

print(f"✅ Data loaded successfully!")
print(f"📊 Shape: {df.shape}")
print(f"📅 Date range: {df.index.min()} to {df.index.max()}")
print(f"📈 Total periods: {len(df)}")
df.head(10)

In [ ]:
# Data info
print("=" * 60)
print("📋 DATA INFORMATION")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("📊 DATA TYPES")
print("=" * 60)
print(df.dtypes)

print("\n" + "=" * 60)
print("🔍 NULL VALUES")
print("=" * 60)
print(df.isnull().sum())

In [ ]:
# Statistical summary
print("=" * 60)
print("📈 STATISTICAL SUMMARY")
print("=" * 60)
df.describe()

In [ ]:
# Check for duplicates
duplicates = df.index.duplicated().sum()
print(f"🔄 Duplicate timestamps: {duplicates}")

if duplicates > 0:
    print("\n⚠️ Duplicate timestamps found:")
    print(df[df.index.duplicated(keep=False)].sort_index().head())
    df = df[~df.index.duplicated(keep='first')]
    print(f"\n✅ Removed duplicates. New shape: {df.shape}")

In [ ]:
# Data quality check
print("=" * 60)
print("🔍 DATA QUALITY CHECK")
print("=" * 60)

# Check for zero or negative values
for col in ['open', 'high', 'low', 'close', 'volume']:
    if col in df.columns:
        invalid = (df[col] <= 0).sum()
        print(f"{col}: {invalid} invalid values (<=0)")

# Check OHLC logic
invalid_ohlc = ((df['high'] < df['low']) | 
               (df['high'] < df['open']) | 
               (df['high'] < df['close']) |
               (df['low'] > df['open']) | 
               (df['low'] > df['close'])).sum()
print(f"\n⚠️ Invalid OHLC relationships: {invalid_ohlc}")

if invalid_ohlc > 0:
    print("\nInvalid rows:")
    invalid_mask = ((df['high'] < df['low']) | 
                   (df['high'] < df['open']) | 
                   (df['high'] < df['close']) |
                   (df['low'] > df['open']) | 
                   (df['low'] > df['close']))
    print(df[invalid_mask].head())